In [80]:
import numpy as np 
import pandas as pd
import os 
from pathlib import Path 
import json 

In [81]:
# BASE=Path('RAW_DATA')
CORRESPONDENCE_CSV = 'bns_correspondence.csv'
COMPARATIVE_CSV = 'BNS_IPC_Comparative.csv'
OUT ='ipc_bns_outputs'
TEXTS_JSON =  'statutory_texts.json'

In [82]:
df=pd.read_csv(CORRESPONDENCE_CSV)

In [83]:
df.head()

,act,section,title,ipc_equivalent,chapter_key,chapter,source,law_version
0,BNS,3(5),Acts done by several persons in furtherance of...,IPC 34,GEN,"General explanations, abetment, conspiracy, at...",NCRB CyTrain BNS-IPC correspondence table,IN-CRIM-2024-07-01
1,BNS,3(6),Act criminal by reason of being done with crim...,IPC 35,GEN,"General explanations, abetment, conspiracy, at...",NCRB CyTrain BNS-IPC correspondence table,IN-CRIM-2024-07-01
2,BNS,45,Abetment of a thing,IPC 107,GEN,"General explanations, abetment, conspiracy, at...",NCRB CyTrain BNS-IPC correspondence table,IN-CRIM-2024-07-01
3,BNS,46,Abettor,IPC 108,GEN,"General explanations, abetment, conspiracy, at...",NCRB CyTrain BNS-IPC correspondence table,IN-CRIM-2024-07-01
4,BNS,49,Punishment of abetment where act abetted is co...,IPC 109,GEN,"General explanations, abetment, conspiracy, at...",NCRB CyTrain BNS-IPC correspondence table,IN-CRIM-2024-07-01


In [84]:
df=df[['section','ipc_equivalent']]

In [85]:
df.head()

,section,ipc_equivalent
0,3(5),IPC 34
1,3(6),IPC 35
2,45,IPC 107
3,46,IPC 108
4,49,IPC 109


In [86]:
df['IPC_Section']=df['ipc_equivalent'].str.replace('IPC','',regex=False).str.strip()

In [87]:
df=df.rename(columns={'section':'BPC_Section'})

In [88]:
df.head()

,BPC_Section,ipc_equivalent,IPC_Section
0,3(5),IPC 34,34
1,3(6),IPC 35,35
2,45,IPC 107,107
3,46,IPC 108,108
4,49,IPC 109,109


In [89]:
df.drop(columns=['ipc_equivalent'],inplace=True)

In [90]:
df.head()

,BPC_Section,IPC_Section
0,3(5),34
1,3(6),35
2,45,107
3,46,108
4,49,109


In [91]:
df1=pd.read_csv(COMPARATIVE_CSV)

In [92]:
df1.head()

,BNS_Section,BNS_Title,Status,IPC_Section,IPC_Title
0,1(1),"Short title, commencement and application",NaN,1,Title and extent of operation of the Code
1,1(2),"Short title, commencement and application",New Sub-Section,NaN,NaN
2,1(3),"Short title, commencement and application",NaN,2,Punishment of offences committed within India
3,1(4),"Short title, commencement and application",NaN,3,"Punishment of offences committed beyond, but w..."
4,1(5),"Short title, commencement and application",NaN,4,Extension of Code to extra-territorial offences


In [93]:
df1=df1[['BNS_Section','IPC_Section']]

In [94]:
df1.head()

,BNS_Section,IPC_Section
0,1(1),1
1,1(2),NaN
2,1(3),2
3,1(4),3
4,1(5),4


In [97]:
import pandas as pd

# -------------------------------------------------------------------
# 1. Clean & Standardize Both DataFrames
# -------------------------------------------------------------------
# Keep only the first two columns of df and rename them explicitly
df = df.iloc[:, :2]
df.columns = ['BNS_Section', 'IPC_Section']

# Keep only the BNS_Section and IPC_Section columns of df1 if it has extra columns
df1 = df1[['BNS_Section', 'IPC_Section']]

# Clean string formatting (remove trailing spaces, strip 'IPC' prefix)
df['BNS_Section'] = df['BNS_Section'].astype(str).str.strip()
df['IPC_Section'] = df['IPC_Section'].astype(str).str.replace('IPC', '', regex=False).str.strip()

df1['BNS_Section'] = df1['BNS_Section'].astype(str).str.strip()
df1['IPC_Section'] = df1['IPC_Section'].astype(str).str.replace('IPC', '', regex=False).str.strip()

# Drop duplicate rows within individual datasets
df = df.drop_duplicates(subset=['BNS_Section', 'IPC_Section'])
df1 = df1.drop_duplicates(subset=['BNS_Section', 'IPC_Section'])

# -------------------------------------------------------------------
# 2. Outer Merge on BNS_Section
# -------------------------------------------------------------------
merged = pd.merge(
    df,
    df1,
    on='BNS_Section',
    how='outer',
    suffixes=('_df1', '_df2')
)

# -------------------------------------------------------------------
# 3. Consolidate into Master DataFrame
# -------------------------------------------------------------------
def consolidate_row(row):
    ipc1 = row['IPC_Section_df1']
    ipc2 = row['IPC_Section_df2']
    
    valid1 = pd.notna(ipc1) and str(ipc1).lower() != 'nan'
    valid2 = pd.notna(ipc2) and str(ipc2).lower() != 'nan'
    
    # Present in both and agree
    if valid1 and valid2 and ipc1 == ipc2:
        return pd.Series([ipc1, 'Agreed'])
    
    # Present in both but disagree / conflict
    elif valid1 and valid2 and ipc1 != ipc2:
        return pd.Series([f"{ipc1} / {ipc2}", 'Conflict (Needs Review)'])
    
    # Present only in dataset 1
    elif valid1:
        return pd.Series([ipc1, 'Dataset 1 Only'])
    
    # Present only in dataset 2
    else:
        return pd.Series([ipc2, 'Dataset 2 Only'])

# Apply logic across all rows
merged[['Final_IPC_Section', 'Source_Note']] = merged.apply(consolidate_row, axis=1)

# Create clean master DataFrame
master_df = merged[['BNS_Section', 'Final_IPC_Section', 'Source_Note']].drop_duplicates()

# -------------------------------------------------------------------
# 4. Summary Output
# -------------------------------------------------------------------
print("=== Consolidation Summary ===")
print(master_df['Source_Note'].value_counts())

master_df.head(10)

=== Consolidation Summary ===
Source_Note
Dataset 2 Only             467
Agreed                     101
Dataset 1 Only              46
Conflict (Needs Review)      4
Name: count, dtype: int64


,BNS_Section,Final_IPC_Section,Source_Note
0,1(1),1,Dataset 2 Only
1,1(2),NaN,Dataset 2 Only
2,1(3),2,Dataset 2 Only
3,1(4),3,Dataset 2 Only
4,1(5),4,Dataset 2 Only
5,1(6),5,Dataset 2 Only
6,10,72,Dataset 2 Only
7,100,299,Agreed
8,101,300,Agreed
9,102,301,Dataset 2 Only


In [ ]:
RELATION_TYPES = [
    'EXACT',
    'MATERIAL_MODIFICATION',
    'SCOPE_EXPANDED',
    'SCOPE_NARROWED',
    'PUNISHMENT_ALTERED',
    'MERGED',
    'SPLIT',
    'NEW_PROVISION',
    'REPEALED_OR_OMITTED',
    'NO_DIRECT_CORRESPONDENCE',
    'UNCERTAIN',
]